### Librerías a emplear
Pandas - Para manipulación tabular de datos

In [2]:
import pandas as pd

### DATASET - ROTURAS

In [3]:
#Se hace carga del archivo excel correspondiente a la base de datos de roturas y fugas
roturas_df = pd.read_excel("../database/02_ROTURAS_RED_PRIMARIA_SECUNDARIA_SAN_MARTIN.xlsx", sheet_name="ROTURAS_RED_PRIMARIA")

roturas_df.head()

,CONEXION,SECTOR,CDESTIPSER,MES,AÑO,CATEGORIA,fecha_inicio,hora_inicio,fecha_fin,hora_fin,latitud,longitud
0,912570,S74,AG: Conexiones Domiciliarias ...,12,2025,Rotura Tuberia Matriz,2025-12-23,1960-01-01 06:05:00,2025-12-25,1960-01-01 21:19:06.568,-6.478684,-76.357285
1,117989,S74,AG: Conexiones Domiciliarias ...,7,2025,Rotura Tuberia Matriz,2025-07-09,1960-01-01 15:46:00,2025-07-10,1960-01-01 21:17:56.096,-6.509906,-76.356666
2,218970,S74,AG: Conexiones Domiciliarias ...,7,2025,Rotura Tuberia Matriz,2025-07-11,1960-01-01 17:07:00,2025-07-13,1960-01-01 14:05:27.964,-6.510675,-76.370133
3,924804,S74,AG: Conexiones Domiciliarias ...,7,2025,Rotura Tuberia Matriz,2025-07-29,1960-01-01 08:10:00,2025-07-31,1960-01-01 20:10:41.624,-6.500371,-76.366108
4,937439,S74,AG: Conexiones Domiciliarias ...,9,2025,Rotura Tuberia Matriz,2025-09-21,1960-01-01 12:45:00,2025-09-24,1960-01-01 10:13:52.188,-6.494526,-76.382672


In [146]:
#Analizamos los campos disponibles en la tabla
roturas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 726 entries, 0 to 725
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   CONEXION      726 non-null    int64         
 1   SECTOR        726 non-null    str           
 2   CDESTIPSER    726 non-null    str           
 3   MES           726 non-null    int64         
 4   AÑO           726 non-null    int64         
 5   CATEGORIA     726 non-null    str           
 6   fecha_inicio  726 non-null    datetime64[us]
 7   hora_inicio   726 non-null    datetime64[us]
 8   fecha_fin     726 non-null    datetime64[us]
 9   hora_fin      726 non-null    datetime64[us]
 10  latitud       726 non-null    float64       
 11  longitud      726 non-null    float64       
dtypes: datetime64[us](4), float64(2), int64(3), str(3)
memory usage: 68.2 KB


### Vamos a hacer una limpieza de datos en función de las variables

* Encontrar cantidad de registros duplicados en la variable conexión, tipo de rotura (Categoría de rotura) y fecha de registro.
En este caso consideramos duplicidad en caso de coincidencia de las 3 variables anteriormente señaladas.

In [147]:
print("Número de registros duplicados: ", roturas_df.duplicated(subset=["CONEXION", "CATEGORIA", "fecha_inicio"]).sum())


#Eliminamos los registros duplicados a partir de las condiciones y nos quedamos con la primera coincidencia
roturas_df_wtdoble = roturas_df.drop_duplicates(subset=["CONEXION", "CATEGORIA", "fecha_inicio"], keep="first")

# roturas_df_clean.to_excel()

Número de registros duplicados:  4


In [140]:
#obtenemos 5 ejemplos de la distribución de valores de latitud y longitud en los registros
roturas_df_wtdoble[["latitud", "longitud"]].sample(5)

,latitud,longitud
8,-6.475608,-76.360226
324,-6.473762,-76.394436
365,-6.474839,-76.387006
203,-6.478069,-76.361000
367,-6.518519,-76.343508


### Corrección de coordenadas
* Identificar patrón detrás del error y corregirlo
* Sabemos que el dataset corresponde a San Martín y maneja datos aproximados de latitud y longitud. En una búsqueda rápida por internet estos fueron definidos (Hemisferio Sur): latitud: -5;-8 - longitud: -76;-77.
Toda coordenada debe estar dentro del rango señalado

In [ ]:
#Obtenemos todos aquellos registros que no cumplen con la condición

condicion_longitud = (roturas_df_wtdoble["longitud"] > -76.00) | (roturas_df_wtdoble["longitud"] < -77.00)

segmento_corregir = roturas_df_wtdoble[condicion_longitud]

# #para dicho segmento de registros con coordenadas incorrectas, hacemos un intercambio de las mismas
segmento_corregir["latitud_b"] = segmento_corregir["latitud"] #<-- creamos una copia temporal de la columna latitud
segmento_corregir["latitud"] = segmento_corregir["longitud"]
segmento_corregir["longitud"] = segmento_corregir["latitud_b"]

del segmento_corregir["latitud_b"] # <------ eliminamos la copia temporal

# #eliminar todos los registros que no cumplen con la condición (modificamos el original)
roturas_segmentoEliminado = roturas_df_wtdoble.drop(roturas_df_wtdoble[condicion_longitud].index)

""" roturas_segmentoEliminado.to_excel("segmento.xlsx")
clean2.to_excel("clean2.xlsx")  <---- verificamos correcta segmentación de datos erroneos"""


# #Ahora fusionamos el segmento corregido con nuestro dataframe original
roturas_limpio = pd.concat([roturas_segmentoEliminado, segmento_corregir])

# exportamos a excel
# roturas_limpio.to_excel("../database_clean/roturas_clean.xlsx")

### DATASET - FUGAS

In [6]:
fugas_df = pd.read_excel("../database/02_ROTURAS_RED_PRIMARIA_SECUNDARIA_SAN_MARTIN.xlsx", sheet_name="FUGAS_RED_SECUNDARIA")

fugas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11159 entries, 0 to 11158
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   CONEXION      11159 non-null  int64         
 1   SECTOR        11159 non-null  str           
 2   CDESTIPSER    11159 non-null  str           
 3   MES           11159 non-null  int64         
 4   AÑO           11159 non-null  int64         
 5   CATEGORIA     11159 non-null  str           
 6   fecha_inicio  11159 non-null  datetime64[us]
 7   hora_inicio   11159 non-null  datetime64[us]
 8   fecha_fin     11159 non-null  datetime64[us]
 9   hora_fin      11159 non-null  datetime64[us]
 10  latitud       11159 non-null  float64       
 11  longitud      11159 non-null  float64       
dtypes: datetime64[us](4), float64(2), int64(3), str(3)
memory usage: 1.0 MB


In [152]:
print("Número de registros duplicados: ", fugas_df.duplicated(subset=["CONEXION", "CATEGORIA", "fecha_inicio"]).sum())

#Eliminamos los registros duplicados a partir de las condiciones y nos quedamos con la primera coincidencia
fugas_df_wtdoble = fugas_df.drop_duplicates(subset=["CONEXION", "CATEGORIA", "fecha_inicio"], keep="first")
#5 registros de ejemplo para ver la distribución de latitud y longitud
fugas_df_wtdoble[["latitud", "longitud"]].sample(5)

Número de registros duplicados:  299


,latitud,longitud
9723,-6.487605,-76.374467
8912,-6.473147,-76.359143
1069,-6.487759,-76.384839
7194,-6.512828,-76.365489
9496,-6.492219,-76.351557


In [ ]:
#Obtenemos todos aquellos registros que no cumplen con la condición

condicion_longitud = (fugas_df_wtdoble["longitud"] > -76.00) | (fugas_df_wtdoble["longitud"] < -77.00)

segmento_corregir = fugas_df_wtdoble[condicion_longitud]

# #para dicho segmento de registros con coordenadas incorrectas, hacemos un intercambio de las mismas
segmento_corregir["latitud_b"] = segmento_corregir["latitud"] #<-- creamos una copia temporal de la columna latitud
segmento_corregir["latitud"] = segmento_corregir["longitud"]
segmento_corregir["longitud"] = segmento_corregir["latitud_b"]

del segmento_corregir["latitud_b"] # <------ eliminamos la copia temporal

# #eliminar todos los registros que no cumplen con la condición (modificamos el original)
fugas_segmentoEliminado = fugas_df_wtdoble.drop(fugas_df_wtdoble[condicion_longitud].index)



# #Ahora fusionamos el segmento corregido con nuestro dataframe original
fugas_limpio = pd.concat([fugas_segmentoEliminado, segmento_corregir])

# #exportamos a excel
# fugas_limpio.to_excel("../database_clean/fugas_clean.xlsx")


with pd.ExcelWriter("../database_clean/02_ROTURAS_RED_PRIMARIA_SECUNDARIA_SAN_MARTIN_CLEAN.xlsx") as writer:
    roturas_limpio.to_excel(writer, sheet_name="ROTURAS_RED_PRIMARIA", index=False)
    fugas_limpio.to_excel(writer, sheet_name="FUGAS_RED_SECUNDARIA", index=False)